In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
import joblib

# 1. Veriyi tekrar yükle
df = pd.read_csv('../data/processed/nlp_processed_reviews.csv')
df.dropna(subset=['processed_text', 'sentiment'], inplace=True)

# 2. Burada orijinal metinleri (processed_text) ve etiketleri ayırıyoruz
X = df['processed_text']
y = df['sentiment']

# 3. Aynı random_state ile bölüyoruz ki indexler tutarlı olsun
_, X_test_raw, _, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# 4. Eğitilmiş modeli yükle
best_rf = joblib.load('../models/sentiment_model.pkl')
tfidf = joblib.load('../models/tfidf_vectorizer.pkl')

# 5. Metinleri modele sokmak için vektörleştir
X_test_tfidf = tfidf.transform(X_test_raw)

# 6. Hata analizi
df_test = pd.DataFrame({'text': X_test_raw, 'actual': y_test, 'predicted': best_rf.predict(X_test_tfidf)})
errors = df_test[df_test['actual'] != df_test['predicted']]

print("Modelin en çok hata yaptığı örnekler:")
print(errors.head(5))
# İş Odaklı Pipeline Tasarımı (Haftalık Şikayet Özeti)
# Bu kısım dokümantasyonda (README.md) maddeler halinde açıklanacak bir "taslak" olmalı:


Modelin en çok hata yaptığı örnekler:
                                                  text    actual predicted
293  top end bluetooth speaker amazon tap rate righ...  Positive  positive
285                                          good case  Positive  positive
122  remote worked well 30 minute jump around sever...  Positive  positive
344  ive purchased item amazon year rarely write re...   Neutral   neutral
229  little difficult sync first time really used t...  Positive  positive


'\n--- Önerilen Pipeline Taslağı: Haftalık Şikayet Özeti ---\n1. Veri Toplama: Haftalık olarak yeni gelen tüm kullanıcı yorumları veritabanından çekilir.\n2. Filtreleme (Bulanık Mantık): Sadece "Güvenilirlik Skoru" > 70 olan yorumlar işleme alınır.\n3. Duygu Analizi: Eğitilen \'sentiment_model.pkl\' ile yorumlar sınıflandırılır.\n4. Kümeleme (Clustering): \'Negative\' sınıfındaki yorumlar üzerinde LDA (Latent Dirichlet Allocation) \n   veya Keyword Extraction (anahtar kelime çıkarma) çalıştırılır.\n5. Raporlama: En sık geçen negatif kelimeler (örn: \'şarj\', \'ekran\', \'donma\') \n   Otomatik bir e-posta veya Slack bildirimi ile ilgili ürün ekibine raporlanır.\n'

"""
--- Önerilen Pipeline Taslağı: Haftalık Şikayet Özeti ---
1. Veri Toplama: Haftalık olarak yeni gelen tüm kullanıcı yorumları veritabanından çekilir.
2. Filtreleme (Bulanık Mantık): Sadece "Güvenilirlik Skoru" > 70 olan yorumlar işleme alınır.
3. Duygu Analizi: Eğitilen 'sentiment_model.pkl' ile yorumlar sınıflandırılır.
4. Kümeleme (Clustering): 'Negative' sınıfındaki yorumlar üzerinde LDA (Latent Dirichlet Allocation) 
   veya Keyword Extraction (anahtar kelime çıkarma) çalıştırılır.
5. Raporlama: En sık geçen negatif kelimeler (örn: 'şarj', 'ekran', 'donma') 
   Otomatik bir e-posta veya Slack bildirimi ile ilgili ürün ekibine raporlanır.
"""